[← 03 - JOINs](<03 - JOINs (INNER vs LEFT).ipynb>) · [Course Overview](<00 - Course Overview.ipynb>)

# 04 - Subqueries to CTEs

The join in `03 - JOINs` was two tables. Real questions often need several steps: filter, then aggregate, then join the result to something else. Nested subqueries can do this, but they get unreadable fast. A Common Table Expression (CTE) fixes that by giving each step a name.

> **By the end of this notebook you'll be able to:** write a `WITH` CTE, stack several of them into a readable sequence, and know when a temp table is the better tool instead.

## 1. Basic syntax

A CTE starts with `WITH`, a name, `AS`, and a query in parentheses. The main `SELECT` below it references that name like a regular table. Here, `high_value_customers` finds every customer whose orders add up to more than $50,000:

**Example:**

In [ ]:
WITH high_value_customers AS (
    SELECT CustomerID, SUM(TotalDue) AS "Total Spent"
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
    HAVING SUM(TotalDue) > 50000
)
SELECT *
FROM high_value_customers
ORDER BY "Total Spent" DESC;

## 2. Stacking CTEs

The real value shows up once you chain several steps, each one named, each one building on the last. Here we find high value customers, then join back to `Sales.Customer` to get their account number:

**Example:**

In [ ]:
WITH high_value_customers AS (
    SELECT CustomerID, SUM(TotalDue) AS "Total Spent"
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
    HAVING SUM(TotalDue) > 50000
),
customer_summary AS (
    SELECT c.CustomerID, c.AccountNumber, hvc."Total Spent"
    FROM Sales.Customer c
    INNER JOIN high_value_customers hvc
        ON c.CustomerID = hvc.CustomerID
)
SELECT *
FROM customer_summary
ORDER BY "Total Spent" DESC;

## 3. Why this beats a nested subquery

You could write the query above as nested subqueries instead, same result, same execution plan. The difference is entirely in how you read it:

```sql
SELECT *
FROM (
    SELECT c.CustomerID, c.AccountNumber, hvc."Total Spent"
    FROM Sales.Customer c
    INNER JOIN (
        SELECT CustomerID, SUM(TotalDue) AS "Total Spent"
        FROM Sales.SalesOrderHeader
        GROUP BY CustomerID
        HAVING SUM(TotalDue) > 50000
    ) hvc ON c.CustomerID = hvc.CustomerID
) customer_summary
ORDER BY "Total Spent" DESC;
```

To understand the nested version, you read inside-out, starting from the deepest subquery. The CTE version reads top to bottom, in the order the logic actually runs:

![Reading a nested subquery vs. a stack of CTEs](graphics/04_nested_vs_cte.png)

*Diagram source: `graphics/04_nested_vs_cte.mmd`*

## 4. Anti-pattern: don't nest CTEs inside each other

CTEs are defined one after another under a single `WITH`, not nested inside one another:

```sql
-- This errors
WITH outer_cte AS (
    WITH inner_cte AS (SELECT ...)
    SELECT * FROM inner_cte
)
```

List every CTE under one `WITH`, separated by commas, in the order they're needed, the way `high_value_customers` and `customer_summary` are above.

## 5. When to reach for a temp table instead

A CTE isn't materialized by default. If you reference the same CTE more than once in a query, SQL Server may evaluate it more than once. If you need to reference an expensive intermediate result several times, or want to index it, a temp table (`SELECT ... INTO #staging`) is often the more predictable tool. Use a CTE for readability, a temp table when re-evaluation is actually a problem.

## What's Next

CTEs can also reference themselves to walk hierarchical data (an org chart, a folder tree), that's a deeper topic on its own and worth reading about once the syntax above feels natural. Next: `05 - Window Functions, Part 1`, where you keep every detail row *and* get an aggregate, something neither `GROUP BY` nor a CTE does on its own.

---

[← 03 - JOINs](<03 - JOINs (INNER vs LEFT).ipynb>) · [Course Overview](<00 - Course Overview.ipynb>) · [05 - Window Functions, Part 1 →](<05 - Window Functions, Part 1.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).